In [2]:
import sys
from pathlib import Path

# 1. Gestion du chemin racine du projet (PROJET_MS)
BASE_DIR = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.append(str(BASE_DIR))

import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, 
    f1_score, roc_auc_score, classification_report, confusion_matrix
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from database.connection import get_engine

# =========================================================
# 2. Re-extraction des données selon le Schéma exact SQL
# =========================================================
engine = get_engine()

query = """
WITH stats_sinistres AS (
    SELECT 
        contrat,
        COUNT(DISTINCT sinistre) AS nb_sinistres,
        COALESCE(SUM(montant_des_soins), 0) AS total_soins,
        COALESCE(SUM(montant_rembourse), 0) AS total_rembourse,
        COALESCE(AVG(montant_rembourse), 0) AS cout_moyen_rembourse
    FROM fact_sinistre
    GROUP BY contrat
),
stats_cotisations AS (
    SELECT 
        contrat,
        COALESCE(SUM(montant_cotisation_payee), 0) AS total_cotisations
    FROM fact_cotisation
    GROUP BY contrat
),
stats_ayants_droit AS (
    SELECT 
        contrat,
        COUNT(DISTINCT code_membre) AS nb_ayants_droit
    FROM dim_membre
    GROUP BY contrat
)
SELECT 
    c.contrat AS code_contrat,
    c.zone,
    COALESCE(c.produit, 'Non Précisé') AS produit,
    COALESCE(c.montant_du_pret, 400000) AS montant_pret,
    COALESCE(c.duree_du_contrat, 8) AS duree_pret_mois,
    COALESCE(c.montant_total_de_cotisation_a_collecter_sur_ce_contrat, 12000) AS cotisation_due,
    COALESCE(m.nb_ayants_droit, 1) AS nb_ayants_droit,
    COALESCE(s.nb_sinistres, 0) AS nb_sinistres,
    COALESCE(s.total_soins, 0) AS total_soins,
    COALESCE(s.total_rembourse, 0) AS total_rembourse,
    COALESCE(s.cout_moyen_rembourse, 0) AS cout_moyen_rembourse,
    COALESCE(k.total_cotisations, 0) AS total_cotisations
FROM dim_contrat c
LEFT JOIN stats_sinistres s ON c.contrat = s.contrat
LEFT JOIN stats_cotisations k ON c.contrat = k.contrat
LEFT JOIN stats_ayants_droit m ON c.contrat = m.contrat;
"""

df = pd.read_sql(query, engine)

# =========================================================
# 3. Calcul du Score Actuariel & Seuil Adaptatif
# =========================================================
df["ratio_sp"] = df.apply(
    lambda r: r["total_rembourse"] / r["cotisation_due"] if r["cotisation_due"] > 0 
    else (5.0 if r["total_rembourse"] > 0 else 0.0), axis=1
)

df['score_sp'] = (df['ratio_sp'] / 1.5).clip(upper=1.0) * 100
df['score_freq'] = (df['nb_sinistres'] / 3.0).clip(upper=1.0) * 100
df['score_plafond'] = (df['total_rembourse'] / 400000.0).clip(upper=1.0) * 100

df['score_risque'] = (
    0.50 * df['score_sp'] + 
    0.30 * df['score_freq'] + 
    0.20 * df['score_plafond']
).round(1)

# Seuil Adaptatif pour éviter l'absence de classe 1
seuil_60_count = (df['score_risque'] >= 60).sum()
if seuil_60_count >= 5:
    seuil_retenu = 60.0
else:
    p75 = df['score_risque'].quantile(0.75)
    seuil_retenu = max(25.0, float(p75))

df['cible_risque_eleve'] = (df['score_risque'] >= seuil_retenu).astype(int)

print(f"📊 Nombre de contrats chargés : {len(df)}")
print(f"🎯 Seuil de risque retenu : {seuil_retenu}")
print("📉 Distribution de la cible :")
print(df['cible_risque_eleve'].value_counts())

# Sauvegarde du dataset d'entraînement
data_path = BASE_DIR / "data" / "processed" / "dataset_ml_risque.csv"
data_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(data_path, index=False)

# =========================================================
# 4. Séparation X (Souscription) / y (Cible Risk)
# =========================================================
feature_cols = [
    'zone', 
    'produit', 
    'montant_pret', 
    'duree_pret_mois', 
    'cotisation_due', 
    'nb_ayants_droit'
]

X = df[feature_cols]
y = df['cible_risque_eleve']

num_features = ['montant_pret', 'duree_pret_mois', 'cotisation_due', 'nb_ayants_droit']
cat_features = ['zone', 'produit']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
    ]
)

stratify_option = y if y.nunique() > 1 and y.value_counts().min() >= 2 else None
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=stratify_option
)

# =========================================================
# 5. Entraînement des Modèles & Sauvegarde Pipeline
# =========================================================
models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

results = []
models_dir = BASE_DIR / "models"
models_dir.mkdir(parents=True, exist_ok=True)

for name, model in models.items():
    pipe = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', model)])
    pipe.fit(X_train, y_train)
    
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1] if hasattr(pipe, "predict_proba") else None
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_proba) if (y_proba is not None and len(np.unique(y_test)) > 1) else np.nan
    
    results.append({
        'Modèle': name,
        'Accuracy': round(acc, 3),
        'Précision': round(prec, 3),
        'Rappel': round(rec, 3),
        'F1-Score': round(f1, 3),
        'ROC-AUC': round(auc, 3)
    })
    
    # Modèle sauvegardé pour l'application Streamlit
    if name == 'Random Forest':
        joblib.dump(pipe, models_dir / 'best_risk_model.pkl')

print("\n🏆 PERFORMANCES DES MODÈLES :")
df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))

print("\n✅ Fichier 'best_risk_model.pkl' généré avec succès dans le dossier /models !")

📊 Nombre de contrats chargés : 5788
🎯 Seuil de risque retenu : 60.0
📉 Distribution de la cible :
cible_risque_eleve
0    5434
1     354
Name: count, dtype: int64

🏆 PERFORMANCES DES MODÈLES :
             Modèle  Accuracy  Précision  Rappel  F1-Score  ROC-AUC
Logistic Regression     0.653      0.085   0.479     0.145    0.641
      Random Forest     0.880      0.160   0.225     0.187    0.616
  Gradient Boosting     0.939      0.500   0.042     0.078    0.625

✅ Fichier 'best_risk_model.pkl' généré avec succès dans le dossier /models !
